# Clonal Insights snippets

<b>This notebook is set up to work with Clonal Insights h5 files, generated with v3 chemistry. This notebook will only work with Mosaic versions 3.17 and higher.</b>


<b>Objective</b>: To showcase steps to adjust output from the Clonal Insights report and generate a new report, and perform CNV analysis.<br>

<b>Sections:</b>
1. Setup
3. DNA Analysis
4. CNV Analysis
5. Protein Clustering
7. Export and Save Data

<b>Not shown:</b>
All available methods and options - documented [here](https://missionbio.github.io/mosaic/index.html)<br>

## Setup 

<b>Topics covered</b><br>
1. Load dependencies
2. Load data


### Load Dependencies
<b>NOTE:</b> importing dependencies can sometimes take a couple of minutes.

In [ ]:
import warnings
warnings.filterwarnings('ignore')

# Import mosaic libraries
import missionbio.mosaic as ms
from missionbio.config.config import Config     
from missionbio.mosaic.assay import _Assay

# Import graph_objects from the plotly package to display figures when saving the notebook as an HTML
import plotly as px
import plotly.graph_objects as go

# Import additional packages for specific visuals
import matplotlib.pyplot as plt
import plotly.offline as pyo
pyo.init_notebook_mode()
import pandas as pd
import numpy as np
import seaborn as sns
from missionbio.plotting.multimap import MultiMap
from missionbio.plotting.heatmap import Heatmap
import plotly.graph_objects as go
from IPython.display import display, HTML
from sklearn.decomposition import PCA

# Import dependencies to create a new report
from pathlib import Path 
from missionbio.tertiary.config.prepare import DEFAULTS
from missionbio.tertiary.step.report import create_single_patient_report, create_cohort_report

# Read in the configuration file
config = Config.read({}, DEFAULTS)

# Note: when exporting the notebook as an HTML, plots that use the "go.Figure(fig)" command are saved

### Load the data

In [ ]:
# Specify the h5 file to be used in this analysis: h5path = '/path/to/h5/file/test.h5'
# If working with Windows, an 'r' may need to be added before the path: h5path = r'/path/to/h5/file/test.h5'
h5path = './Example.h5'
sample = ms.load(h5path)

## DNA Analysis

Review variants included in the Clonal Insights report and remove unwanted variants.

In [ ]:
# List all DNA variants
sample.dna.ids()

In [ ]:
# Blacklist any variants you would like to remove
blacklist_vars = ['chr2:25469913:C/T','chr2:25457242:C/T']
sample.dna = sample.dna.drop(blacklist_vars)

In [ ]:
# Check that variants were dropped from the sample
sample.dna.ids()

## CNV Analysis


<b>Topics covered</b>
1. Amplicon and cell filtering and ploidy estimation
2. Visualization of ploidy across subclones present

In [ ]:
# Get gene names for amplicons
sample.cnv.get_gene_names();

### CNV workflow

This workflow will normalize all reads and filter amplicons/cells based on the settings set at the beginning of the workflow: <br>
1. <b> Amplicon completeness: </b> refers to the minimum percentage of cells that must have reads greater than or equal to the minimum read depth set. By default this is set to 50. <br>
2. <b> Amplicon read depth: </b> refers to the minimum read depth for each amplicon-barcode combination to not be considered missing. By default this is set to 10. <br>
3. <b>Mean cell read depth: </b> refers the minimum mean read depth for a cell to be included in the analysis, otherwise the cell will be removed. By default this is set to 40.<br>
4. <b>Diploid clone in DNA: </b> refers to which subclone is used as the true diploid population. All ploidy estimates will be calculated in relation to this diploid population. We recommend setting this to the 'WT' population or earliest clone present. <br>
    
<b> NOTE: </b> If large copy number events are expected, Amplicon Completeness and Amplicon Read Depth may need to be reduced.

Once the above filters are set, the visualizations can be changed. <br>
1. <b> Plot:</b> Can be changed from Heatmap positions, to Heatmap genes, Line-plot positions, Line-plot genes. <br>
2. <b> Clone for line plot:</b> If one of the line-plot visualizations is selected, only one clone can be shown at a time. This determines which one is plotted. <br>
3. <b> X-axis features: </b> To plot a subset of the data (chromosomes or genes), select which chromosomes/genes should be plotted with this function. Chromosomes can be selected for 'positions' type plots, and Genes can be selected for 'genes' type plots.

More on this workflow can be found [here](https://missionbio.github.io/mosaic/pages/missionbio.mosaic.workflows.copy_number.CopyNumber.html).

In [ ]:
# CNV workflow to filter, normalize and estimate ploidy
wfc = ms.workflows.CopyNumber(sample)
wfc.run()

<b>NOTE:</b> When continuing onto the next piece of the notebook, the workflow will stop working.

In [ ]:
# Heatmap with the features ordered by the default amplicon order
# To plot a subset of chromosome, the chromosomes can be put in list format in the features argument
fig = sample.cnv.heatmap('ploidy', features='positions') #features=['7', '17', '20']

# Optionally, restrict the range of ploidy values based on observed/expected CNV events (commented out)
#fig.layout.coloraxis.cmax = 4
#fig.layout.coloraxis.cmin = 0

# Optionally, change the size of the figure:
#fig.layout.width = 1600
#fig.layout.height = 1500

go.Figure(fig)

In [ ]:
# Heatmap with the features grouped by the genes
# To plot just a subset of genes, put them in list format for features
fig = sample.cnv.heatmap('ploidy', features='genes', convolve=1) #features=["ASXL1", "EZH2",'TP53']

# Optionally, update the separating lines to be black
#for shape in fig.layout.shapes:
#    shape.line.color = '#000000'

go.Figure(fig)

In [ ]:
# Show heatmap with convolve and subclustering turned off
bars = sample.cnv.clustered_barcodes('ploidy', subcluster=False)

# This is useful to create "convolved" heatmaps which are easier to interpret
# With the subclustering off and convolve=20, the noise will be reduced and real signals will be easier to visualize
fig = sample.cnv.heatmap('ploidy', bars_order=bars, convolve=20)
fig.layout.width = 900
fig.show()

## Protein Analysis

Review protein clustering and recluster by modifying parameters.

In [ ]:
# load default config and update parameters if desired
config["tertiary.rna.normalize.min_reads_per_cell"] = 10 
config["tertiary.rna.cluster.n_neighbors"] = 40

## Export and Save Data

In this section the data can export a filtered .h5 file, which will contain all new labels/layers, and contain only the filtered barcodes/cells remaining. All data can be exported (row attributes, column attributes and layers) for every assay (DNA, CNV and Protein) into easily parsable .csv tables. Additionally, an updated .html report can be generated based on the analysis performed in this notebook.

In [ ]:
# Save new h5 file that includes only the final, cleaned dataset
ms.save(sample, 'FilteredData.h5')

In [ ]:
# Export data into csv formats
ms.to_zip(sample, 'filename')

In [ ]:
# make report
create_single_patient_report(
    sample,
    location=Path("CI_output/report.html"),
    config=config,
)        